## Support Displacement of a Frame


### Solve Question 7.3 in Kassimali

Determine the joint displacements, member end forces, and support reactions for a 
continuous beam, due to the combined effect of the uniformly distributed load shown and the settlements of 45 mm and 15 mm, respectively, of supports 3 and 4. 

Using the DSM, model the structure as **frame** elements, NOT **beam** elements, you should get the same answer as the textbook.


<div style="text-align:center;">
  <img src="../../Lectures/L9/assets/L1_Example2.png" style="width:95%;">
</div>

In [ ]:
# -------------------------
# Import Libraries and Helper Functions
# -------------------------
# This slide imports the required Python libraries and adds the
# course helper-code directory to the Python path.

import sys
import os
import numpy as np
import pandas as pd

# Get current notebook directory
current_dir = os.getcwd()

# Go up two levels, then into Code/L8
helpers_path = os.path.abspath(
    os.path.join(current_dir, "..", "..", "Code", "L8")
)

sys.path.append(helpers_path)

In [ ]:
import numpy as np

from helper_funcs_dsm import (
    assemble_global_stiffness_and_fef,
    partition_system,
    assemble_global_displacements,
    assemble_global_forces
)
from helper_funcs_output import (
    print_element,
    print_dsm_results,
    print_matrix_scaled
)


In [ ]:
# -------------------------
# Problem data (hard-coded)
# -------------------------
E = 200e6          # kPa
I = 102e-6         # m^4
L = 8.0            # m
A = #Set to anything, doesnt matter for a beam, no axial force

In [ ]:
# -------------------------
# Member 1
# -------------------------
k1 = #TODO

# Transformation matrix
T1 = #TODO

# Mapping
m1 = #TODO

In [ ]:
# Local fixed-end vector for member 1
Qf1 = #TODO

In [ ]:
# -------------------------
# Member 2
# -------------------------
k2 = #TODO

# Transformation matrix
T2 = #TODO

# Mapping
m2 = #TODO

In [ ]:
# Local fixed-end vector for member 2
Qf2 = #TODO

In [ ]:
# -------------------------
# Member 3
# -------------------------
k3 = #TODO

# Transformation matrix
T3 = #TODO

# Mapping
m3 = #TODO

In [ ]:
# Local fixed-end vector for member 1 (given by the example result)
Qf3 = #TODO

In [ ]:
# -------------------------
# Initialize Global System
# -------------------------
# This slide sets up the global variables needed for assembly of the
# Direct Stiffness Method system.

k_list = [k1, k2, k3]
T_list = [T1, T2, T3]
Qf_list = [Qf1, Qf2, Qf3]
map_list = [m1, m2, m3]   # 1-based DOF maps

# Global system size (still using 1-based maps here)
ndof = int(np.max(np.concatenate(map_list)))

# Initialize Applied Load
F_global = np.zeros(ndof, dtype=float)

# Initialize Global Displacement
u_global = np.zeros(ndof, dtype=float)

In [ ]:
# -------------------------
# User-Specified Boundary Conditions
# -------------------------
# This slide defines the external loads and support conditions.

# Applied Load
# F_global[0] = 0.0

# Global Displacement (0-indexed)
#TODO

# Restrained DOFs
dof_restrained_1based = #TODO

# # Fictitious restrained DOFs
# dof_fictitious_1based = np.array([], dtype=int)

# # Combine and sort
# dof_restrained_1based = np.sort(
#     np.concatenate((dof_restrained_1based, dof_fictitious_1based))
# )

In [ ]:
# -------------------------
# Assemble Global Stiffness System
# -------------------------
# This step assembles the global stiffness matrix and the global
# fixed-end force vector from all elements.

K_global, F_fef_global = assemble_global_stiffness_and_fef(
    ndof,
    k_list,
    T_list,
    Qf_list,
    map_list
    )

In [ ]:
# -------------------------
# Display Global Stiffness Matrix
# -------------------------
# Print the assembled global stiffness matrix in a readable format.
# The matrix is scaled for clearer presentation in the slides.

print_matrix_scaled(K_global, scale=1000, decimals=2, col_width=6)

In [ ]:
# -------------------------
# Partition Global System
# -------------------------
# The global stiffness equation is partitioned into free and restrained
# degrees of freedom based on the support conditions.

(
    K_ff,
    K_fr,
    K_rf,
    K_rr,
    f_f,
    f_r,
    u_r,
    f_fef_f,
    f_fef_r,
    free_dofs,
    restrained_dofs,
) = partition_system(
    K_global,
    F_global,
    u_global,
    F_fef_global,
    dof_restrained_1based
)

print(u_r)


In [ ]:
# -------------------------
# Solve Partitioned System
# -------------------------
# Solve for the unknown displacements at the free DOFs,
# compute the reactions at restrained DOFs, and then
# reconstruct the complete global displacement and force vectors.

rhs = f_f - f_fef_f - K_fr @ u_r
u_f = np.linalg.solve(K_ff, rhs)

F_r = K_rf @ u_f + K_rr @ u_r + f_fef_r

u_global = assemble_global_displacements(
    u_f,
    u_r,
    free_dofs,
    restrained_dofs
    )

f_global_complete = assemble_global_forces(
    f_f,
    F_r,
    free_dofs,
    restrained_dofs
    )

In [ ]:
# -------------------------
# Display DSM Results
# -------------------------
# Print the final global displacement vector and force vector.

print_dsm_results(
    u_global,
    f_global_complete,
    dof_restrained_1based,
    disp_in_mm=True,
    member_type="frame"
)

In [ ]:
# -------------------------
# Display Element Results
# -------------------------
# Compute and print the element-level results for each member.

print_element(1, u_global, m1, T1, k1, Qf1, disp_in_mm=True, dec=1, rad_dec=6)
print_element(2, u_global, m2, T2, k2, Qf2, disp_in_mm=True, dec=1, rad_dec=6)
print_element(3, u_global, m3, T3, k3, Qf3, disp_in_mm=True, dec=1, rad_dec=6)

In [ ]:
# -------------------------
# Print Support Reactions
# -------------------------
print(np.round(f_global_complete[dof_restrained_1based - 1], 3))